In [22]:
from langgraph.graph import StateGraph, END, START
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

True

In [23]:
model = ChatGroq(model="llama-3.3-70b-versatile")

In [24]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str
    evaluate:int

In [25]:
def create_outline(state: BlogState) -> BlogState:
    title = state["title"]
    response = model.invoke(f"Create an outline for a blog post titled: {title}")
    state["outline"] = response.content
    return state

In [31]:
def create_blog(state: BlogState) -> BlogState:
    outline = state["outline"]
    response = model.invoke(f"Create a blog post based on the following outline: {outline}")
    state["content"] = response.content
    return state

In [32]:
def evaluate_blog(state: BlogState) -> BlogState:
    content = state["content"]
    response = model.invoke(
    f"""
    Evaluate the following blog post and return ONLY a single integer from 1 to 10.

    Blog:
    {content}
    """
)
    state["evaluate"] = int(response.content.strip())
    return state

In [33]:
graph = StateGraph(BlogState)

graph.add_node("create_outline", create_outline)
graph.add_edge(START, "create_outline")
graph.add_node("create_blog", create_blog)
graph.add_edge("create_outline", "create_blog")
graph.add_node("evaluate_blog", evaluate_blog)
graph.add_edge("create_blog", "evaluate_blog")
graph.add_edge("evaluate_blog", END)
workflow = graph.compile()


In [35]:
initial_state = {"title": "The Future of AI in Healthcare", "outline": "", "content": "", "evaluate": 0}
final_state = workflow.invoke(initial_state)
print(final_state["evaluate"])

8
